# Lesson 2 — Answer Key

For each exercise we show a **wrong** approach (common mistakes) and a **better** approach.

Run the setup cell first, then explore each exercise.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
STRONG_MODEL = "gpt-5.2"
WEAK_MODEL = "gpt-4.1-nano"
print("Client ready")

---
## Exercise 0: Warm-Up — Structured Extraction

### Wrong approach
A vague prompt that doesn't specify the JSON schema. The model returns markdown, extra commentary, or a different structure every time.

In [ ]:
with open("prompting/sherlock-short.txt") as f:
    sherlock_text = f.read()

with open("prompting/sherlock.txt") as f:
    sherlock_long_text = f.read()

print(f"Short text: {len(sherlock_text):,} characters")
print(f"Long text:  {len(sherlock_long_text):,} characters")

# WRONG: vague prompt, no schema, no "JSON only" instruction
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": f"Summarize this text:\n\n{sherlock_text}"},
    ],
    temperature=0,
)
raw = r.choices[0].message.content
print("\nRaw output (first 400 chars):")
print(raw[:400])
print()

# This will almost certainly crash — the model returns prose, not JSON
try:
    data = json.loads(raw)
    print("Parsed OK (lucky!)")
except json.JSONDecodeError:
    print("FAILED to parse as JSON — the model returned prose, not structured data.")

### Better approach
A system message that explains the task, specifies the exact JSON schema, and says "return ONLY valid JSON". The text goes in the user message.

In [ ]:
SYSTEM_PROMPT = """You are a literary analysis assistant. Given a text excerpt, extract structured information and return ONLY valid JSON — no markdown, no commentary, no extra text.

Return a JSON object with these fields:
- "title" (string): the title of the story or book
- "author" (string): the author's full name
- "characters" (array): each element has "name" (string) and "description" (string, 1 sentence)
- "plot_summary" (string): 2-3 sentence summary of what happens in this excerpt
- "setting": object with "locations" (array of strings) and "time_period" (string)"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sherlock_text},
    ],
    temperature=0,
)
raw = r.choices[0].message.content
book_info = json.loads(raw)
print(json.dumps(book_info, indent=2))

---
## Exercise 1: Schema Design — What Would You Extract?

### Wrong approach
Ask the model to "extract info" without specifying the schema. You get inconsistent keys, missing fields, and a different structure every time.

In [ ]:
# WRONG: vague prompt, no schema
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract all the interesting info from this text as JSON:\n\n{sherlock_text}"}],
    temperature=0,
)
raw = r.choices[0].message.content
print("WRONG — vague prompt, model picks its own schema:")
print(raw[:600])
print("\n... (the model invents whatever structure it wants — different every time)")

### Better approach
Design a schema that captures what's interesting about the text, then embed an **example JSON** directly in the prompt so the model knows exactly what shape to produce.

In [ ]:
# BETTER: explicit schema with example JSON in the prompt

SCHEMA_PROMPT = """You are a literary analysis assistant. Extract structured information from the given text and return ONLY valid JSON — no markdown, no commentary.

Use this exact schema (fill in the values based on the text):
{
  "narrator": "name of the narrator",
  "main_characters": [
    {
      "name": "character name",
      "role": "their role in the story (e.g. detective, narrator, client)",
      "traits": ["trait1", "trait2"]
    }
  ],
  "locations": ["place1", "place2"],
  "time_period": "when the story is set",
  "mentioned_cases": [
    {
      "name": "case name or description",
      "details": "brief detail if given"
    }
  ],
  "key_relationships": [
    {
      "between": ["person1", "person2"],
      "nature": "description of the relationship"
    }
  ],
  "mood": "overall emotional tone of the excerpt"
}"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SCHEMA_PROMPT},
        {"role": "user", "content": sherlock_text},
    ],
    temperature=0,
)
raw = r.choices[0].message.content
extracted = json.loads(raw)
print("BETTER — explicit schema with example JSON:")
print(json.dumps(extracted, indent=2))

---
## Exercise 2: Few-Shot Learning — Emotion in Watson's Narration

### Wrong approach
A vague prompt with no guidance on output format or what "emotion" means. The model returns long prose explanations instead of concise labels.

In [ ]:
# WRONG: vague prompt, no format guidance, no examples
r = client.chat.completions.create(
    model=WEAK_MODEL,
    messages=[
        {"role": "system", "content": "What emotions are in this text?"},
        {"role": "user", "content": sherlock_text},
    ],
    temperature=0,
)
print("=== WRONG — vague prompt ===\n")
print(r.choices[0].message.content[:500])
print("\n... (model rambles in prose — no structure, inconsistent granularity)")

### Better approach
A clear system prompt that specifies the output format (JSON array), asks for 1-2 word emotions, and includes **few-shot examples** that teach the model your vocabulary and granularity.

In [ ]:
# BETTER: clear prompt with format spec — zero-shot first

ZERO_SHOT_SYSTEM = """You are a literary analyst. Given a text excerpt, identify 5-6 passages that convey strong emotion or feeling. For each, quote the passage and classify its dominant emotion in 1-2 words.

Return ONLY valid JSON — no markdown, no commentary. Use this format:
[{"passage": "exact quote...", "emotion": "1-2 word emotion"}]"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": ZERO_SHOT_SYSTEM},
        {"role": "user", "content": sherlock_text},
    ],
    temperature=0,
)
zero_shot_result = json.loads(r.choices[0].message.content)

print("=== BETTER — Zero-shot with clear prompt ===\n")
for item in zero_shot_result:
    print(f"  {item['emotion']:20s}  ← \"{item['passage'][:60]}...\"")

# Now add few-shot examples WITH negative examples to steer vocabulary and passage length

FEW_SHOT_SYSTEM = """You are a literary analyst. Given a text excerpt, identify 5-6 passages that convey strong emotion or feeling. For each, quote ONLY the short fragment around the emotional core (not full sentences or paragraphs). Classify the dominant emotion in 1-2 specific words.

GOOD examples (short, focused on the emotional core):
- "alternating from week to week between cocaine and ambition" → restless compulsion
- "seized with a keen desire to see Holmes again" → fond longing
- "with a gibe and a sneer" → cold disdain

BAD examples — do NOT do this:
- Do NOT quote entire paragraphs or multi-sentence passages — extract only the emotionally charged fragment
- Do NOT use generic labels like "positive", "negative", or "emotional" — be specific about WHICH emotion
- Do NOT include factual narration that describes events without feeling (e.g. "One night it was on the twentieth of March" has no emotion)

Return ONLY valid JSON — no markdown, no commentary. Use this format:
[{"passage": "short emotionally charged fragment...", "emotion": "1-2 word emotion"}]"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": FEW_SHOT_SYSTEM},
        {"role": "user", "content": sherlock_text},
    ],
    temperature=0,
)
few_shot_result = json.loads(r.choices[0].message.content)

print("\n=== BETTER — Few-shot with positive + negative examples ===\n")
for item in few_shot_result:
    print(f"  {item['emotion']:20s}  ← \"{item['passage'][:60]}...\"")

In [ ]:
# Compare zero-shot vs few-shot

print("=== Comparison ===\n")
print("Zero-shot emotions:")
for item in zero_shot_result:
    print(f"  {item['emotion']:20s}  ← \"{item['passage'][:65]}...\"")

print("\nFew-shot emotions:")
for item in few_shot_result:
    print(f"  {item['emotion']:20s}  ← \"{item['passage'][:65]}...\"")

print("\nNotice how the few-shot examples steer the model toward the same style of emotion words.")
print("The examples teach vocabulary and granularity — not just the task itself.")

---
## Exercise 3: Chain-of-Thought Prompting

### Wrong approach
Ask for just the answer — no reasoning. On multi-step problems the model often gets it wrong.

In [ ]:
TANK_QUESTION = """A water tank holds 120 liters.
- Tap A fills at 3 liters/minute and is opened at time 0.
- Tap B fills at 5 liters/minute and is opened 10 minutes later.
- A drain empties at 4 liters/minute and is opened 5 minutes after Tap B.
How many minutes after time 0 is the tank completely full?"""

# Work it out:
#   Phase 1 (0–10 min):  3 L/min × 10 = 30 L
#   Phase 2 (10–15 min): (3+5) = 8 L/min × 5 = 40 L → total 70 L
#   Phase 3 (15+ min):   (3+5−4) = 4 L/min, need 50 L more → 50/4 = 12.5 min
#   Total: 15 + 12.5 = 27.5 minutes
TANK_ANSWER = 27.5

# WRONG: direct prompting
print(f"Question: {TANK_QUESTION}")
print(f"Correct answer: {TANK_ANSWER}\n")

print("=== Direct prompting (5 runs, temp=1) ===")
direct_correct = 0
for run in range(1, 6):
    r = client.chat.completions.create(
        model=WEAK_MODEL,
        messages=[
            {"role": "system", "content": "Answer with just the number (in minutes), nothing else."},
            {"role": "user", "content": TANK_QUESTION},
        ],
        temperature=1,
    )
    answer = r.choices[0].message.content.strip()
    try:
        if float(answer) == TANK_ANSWER:
            direct_correct += 1
    except ValueError:
        pass
    print(f"  Run {run}: {answer}")
print(f"\nDirect: {direct_correct}/5 correct")

### Better approach
Ask the model to think step by step, then give the final answer on the last line.

In [ ]:
# BETTER: chain-of-thought prompting
print("=== Chain-of-thought (5 runs, temp=1) ===")
cot_correct = 0
for run in range(1, 6):
    r = client.chat.completions.create(
        model=WEAK_MODEL,
        messages=[
            {"role": "system", "content": "Think step by step. Identify each phase (when taps/drain open), calculate the water level at each transition, then find when it reaches 120 L. On the very last line, write ONLY the final number in minutes."},
            {"role": "user", "content": TANK_QUESTION},
        ],
        temperature=1,
    )
    full = r.choices[0].message.content.strip()
    last_line = full.strip().split("\n")[-1].strip()
    # Extract number from last line
    import re
    nums = re.findall(r"[\d.]+", last_line)
    answer = float(nums[-1]) if nums else -1
    if answer == TANK_ANSWER:
        cot_correct += 1
    print(f"  Run {run}: {answer}  (reasoning: {full[:80]}...)")

print(f"\nCoT: {cot_correct}/5 correct")
print(f"Direct was: {direct_correct}/5 correct")

---
## Exercise 4: Extract → Verify Loop

### Wrong approach
Extract once and trust the result. Weak models miss things and hallucinate — you never know what you got wrong.

In [ ]:
NER_PROMPT_TEMPLATE = """Extract all named entities from the following text and return them as JSON.

Classify each entity as one of: PERSON, ORGANIZATION, LOCATION, DATE, or MISC.

Text: \"{text}\"

Return only valid JSON in this format:
{{
  "entities": [
    {{"text": "...", "type": "...", "start": <char_offset>}}
  ]
}}"""


def extract_entities(text, model=MODEL, temperature=0):
    prompt = NER_PROMPT_TEMPLATE.format(text=text)
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    raw = r.choices[0].message.content
    try:
        return raw, json.loads(raw)
    except json.JSONDecodeError:
        return raw, None

In [ ]:
# WRONG: extract place names once and call it done
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract all place names (cities, streets, countries, regions) from the text. Return ONLY a JSON list of strings, nothing else."},
        {"role": "user", "content": sherlock_long_text},
    ],
    temperature=0,
)
raw = r.choices[0].message.content
try:
    places = parse_json_response(raw)
    print("WRONG — single extraction, no verification:")
    print(json.dumps(places, indent=2))
    print(f"\nGot {len(places)} places. But are they all real? Did we miss any?")
except (json.JSONDecodeError, Exception) as e:
    print(f"WRONG — extraction didn't even return valid JSON:")
    print(f"  {raw[:200]}")

print("\n(With a long text like this, the model is likely to miss some places)")

### Better approach
Extract first, then send the result back with the original text and ask: "Did I miss anything? Did I add anything wrong?" The second pass catches errors the first pass made.

In [ ]:
# BETTER: extract → verify with a second prompt (same model, different task)

import re

def parse_json_response(text):
    """Strip markdown fences if present, then parse JSON."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

# Step 1: Extract places
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract all place names (cities, streets, countries, regions) from the text. Return ONLY a JSON list of strings, nothing else."},
        {"role": "user", "content": sherlock_long_text},
    ],
    temperature=0,
)
places = parse_json_response(r.choices[0].message.content)
print("=== Step 1: Initial extraction ===")
print(json.dumps(places, indent=2))

# Step 2: Verify — same model, but now focused on checking rather than searching
verify_prompt = f"""I extracted these place names from a text. Please check my work.

Extracted places: {json.dumps(places)}

Original text:
\"\"\"{sherlock_long_text}\"\"\"

Check carefully:
1. Are there any places mentioned in the text that I MISSED?
2. Are there any items in my list that are NOT actually places in the text?
3. Are there any that are ambiguous (e.g., "Bohemian" is an adjective, not a place)?

Return ONLY valid JSON (no markdown, no commentary) with this structure:
{{
  "corrected_places": ["place1", "place2", ...],
  "added": ["places that were missing"],
  "removed": ["items that should not be there"],
  "notes": "brief explanation of changes"
}}"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": verify_prompt}],
    temperature=0,
)
verification = parse_json_response(r.choices[0].message.content)

print("\n=== Step 2: Verification ===")
print(json.dumps(verification, indent=2))

print(f"\n--- Summary ---")
print(f"Before: {len(places)} places")
print(f"After:  {len(verification['corrected_places'])} places")
if verification.get("added"):
    print(f"Added:   {verification['added']}")
if verification.get("removed"):
    print(f"Removed: {verification['removed']}")
if verification.get("notes"):
    print(f"Notes:   {verification['notes']}")

---
## Exercise 5: Multi-Document Extraction with Deduplication

### Wrong approach
Extract entities separately, dump them in a list, call it done. You get duplicates everywhere — "Microsoft", "MSFT", "MS" all appear as separate entities.

In [ ]:
ARTICLES = {
    "article_1": (
        "Microsoft announced on Monday that it would acquire GitHub, the world's "
        "largest code-hosting platform, for $7.5 billion in stock. CEO Satya Nadella "
        "called it a strategic move to strengthen Microsoft's developer tools. "
        "The deal is expected to close by the end of 2018."
    ),
    "article_2": (
        "MSFT is buying GitHub for $7.5B — the biggest acquisition since LinkedIn. "
        "Nadella said the deal will let developers 'do their best work.' "
        "GitHub CEO Chris Wanstrath will stay on as a technical fellow. "
        "The EU still needs to approve the acquisition."
    ),
    "article_3": (
        "Developers on Hacker News are divided over the Microsoft-GitHub deal. "
        "Some worry that MS will ruin the platform like it did with Skype. "
        "Others point out that Nat Friedman, the new GitHub CEO, is a respected "
        "open-source advocate from Xamarin. The $7.5 billion price tag raised eyebrows "
        "in San Francisco's tech community."
    ),
}

# WRONG: extract and just concatenate — no deduplication
print("=== WRONG — no deduplication ===\n")
all_entities = []
for name, text in ARTICLES.items():
    raw, parsed = extract_entities(text)
    if parsed:
        for e in parsed["entities"]:
            all_entities.append((name, e))
        print(f"{name}: {len(parsed['entities'])} entities")

print(f"\nTotal raw entities: {len(all_entities)}")
print("\nNotice the duplicates:")
for source, e in all_entities:
    print(f"  [{source}] {e['text']:30s}  {e['type']}")

### Better approach
Use the LLM itself to deduplicate — send all extracted entities and ask it to group them by real-world referent.

In [ ]:
def deduplicate_entities(entities: list[tuple[str, dict]]) -> list[dict]:
    """Use the LLM to group entities that refer to the same real-world thing."""
    # Build a flat list for the prompt
    entity_list = []
    for source, e in entities:
        entity_list.append({"text": e["text"], "type": e["type"], "source": source})

    prompt = f"""Below is a list of named entities extracted from multiple news articles about the same event.
Many of these refer to the same real-world entity using different names (e.g. "Microsoft", "MSFT", "MS" are the same company; "Satya Nadella" and "Nadella" are the same person).

Group them by real-world referent. For each group, pick a canonical name (the most complete/formal version), list aliases, and note which source articles mentioned it.

Entities:
{json.dumps(entity_list, indent=2)}

Return ONLY valid JSON in this format:
[
  {{
    "canonical": "Microsoft",
    "type": "ORGANIZATION",
    "aliases": ["MSFT", "MS"],
    "sources": ["article_1", "article_2", "article_3"]
  }}
]"""

    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = r.choices[0].message.content
    return json.loads(raw)


# BETTER: extract then deduplicate
deduped = deduplicate_entities(all_entities)

print(f"=== BETTER — deduplicated: {len(deduped)} unique entities ===\n")
for group in deduped:
    aliases = ", ".join(group["aliases"]) if group["aliases"] else "(none)"
    sources = ", ".join(group["sources"])
    print(f"  {group['canonical']:25s}  [{group['type']}]")
    print(f"    Aliases: {aliases}")
    print(f"    Sources: {sources}\n")